# Notebook 10: Speculative Decoding

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 01–04 (tokenization, architecture, sampling, decoding algorithms)

---

## Background: The Bottleneck That Wasn't the Model

By 2022, the LLM community had a frustrating problem: GPT-3 and its descendants were extraordinarily capable, but generating text was *agonizingly slow*. Even with fast GPUs, a 70B-parameter model might produce only 10–20 tokens per second. Users of production APIs noticed this viscerally — watching a chatbot type out one word at a time felt like watching someone hunt-and-peck on a keyboard.

The obvious response was to throw more GPUs at the problem. Tensor parallelism helped, but only so much — communication overhead between GPUs became its own bottleneck. Quantization helped too (that's notebook 09), but at the cost of quality. The community wanted something that gave the speed of a small model with the quality of a large one.

### Why Decoding Is Slow: A Memory Bandwidth Problem

Here's the dirty secret: **the GPU is mostly idle during autoregressive decoding.** Each forward pass during the decode phase generates exactly *one* token. That means:

1. Load all model weights from HBM (GPU memory) to SRAM (compute units)
2. Do a tiny matrix-vector multiply (sequence length is usually short relative to model size)
3. Sample one token
4. Wait... then do it all over again

For a 70B model in FP16, that's **140 GB of weights** to stream through memory *for every single token*. The GPU's compute units are barely used — they're just waiting for data to arrive. This is called being **memory bandwidth bound**, not compute bound.

The arithmetic: an A100 has ~2 TB/s memory bandwidth and ~312 TFLOPS of FP16 compute. Loading 140 GB takes ~70ms. But computing a single token's forward pass at batch size 1 requires only ~280 GFLOPs → ~0.9ms of compute. The GPU is spending 99%+ of its time fetching weights, not doing math.

### The Speculative Decoding Insight (2022–2023)

Two independent research threads arrived at essentially the same idea almost simultaneously:

- **Chen et al. (Google, 2023):** "Accelerating Large Language Model Decoding with Speculative Sampling" — the canonical speculative decoding paper
- **Leviathan et al. (Google Brain, 2023):** "Fast Inference from Transformers via Speculative Decoding" — published concurrently, same core algorithm

The insight is elegant: **what if a small, cheap model did most of the guessing, and the big model just checked its work in parallel?**

Here's why this is faster: checking whether K proposed tokens are correct can be done in a *single* forward pass of the large model with sequence length K. That pass takes about the same wall-clock time as generating *one* token autoregressively — because the cost is still dominated by loading weights, not by compute. So if even a fraction of the proposals are accepted, you get multiple tokens per large-model forward pass.

### The Math Behind Acceptance

The algorithm maintains **lossless quality** through a clever rejection sampling scheme. Given:
- Draft model probability: $q(x_t | \text{context})$
- Target model probability: $p(x_t | \text{context})$

A proposed token $\hat{x}_t$ is:
- **Accepted** with probability $\min(1, p(\hat{x}_t) / q(\hat{x}_t))$
- **Rejected** otherwise, and a correction token is sampled from a modified distribution

This ensures the *output distribution* is **exactly** $p$ — the large model's distribution — even though the draft model proposed the tokens. You get the small model's speed with the large model's quality.

### Who Uses This?

By 2024, speculative decoding became standard infrastructure:
- **Google Gemini** uses it in production (Medusa heads variant)
- **Meta LLaMA ecosystem**: speculative decoding is baked into llama.cpp and vLLM
- **Hugging Face**: `assisted_generation` in `transformers` implements draft-verify
- **Apple MLX**: uses speculative decoding for on-device inference
- Typical speedup: **2–3× on decode-heavy workloads** with zero quality degradation

The technique spawned a whole family of variants: Medusa (multiple draft heads on the same model), EAGLE (feature-level draft model), SpecInfer (tree-based speculation), and more. But the core draft-and-verify loop is what everyone builds from.

---

## What You'll Build

In this notebook we'll implement the complete speculative decoding pipeline:
1. **Bigram draft model** — fast, lightweight token predictor
2. **Target model wrapper** — simulates the "oracle" distribution
3. **Draft-verify loop** — the core speculative sampling algorithm
4. **Acceptance rate analysis** — what determines when speculation wins or loses
5. **Speedup model** — when to use speculative decoding and when not to

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import Optional
import time
import random

# Use MPS if on Apple Silicon, otherwise CPU (no GPU needed for this demo)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## Part 1: The Core Algorithm

Before building any models, let's implement the core speculative sampling algorithm. This is the mathematical heart — everything else is scaffolding.

The algorithm (from Chen et al. 2023) for a single speculation step:

```
For each proposed token x̂ from the draft model:
  Get p(x̂) from target model
  Get q(x̂) from draft model  
  Accept with probability min(1, p(x̂) / q(x̂))
  If accepted: keep x̂, continue to next position
  If rejected: sample from (p - q)+ / Z, stop speculation

After all proposals accepted: sample one bonus token from target
```

The `(p - q)+` distribution ("adjusted distribution") ensures we're sampling from exactly the target distribution conditioned on what was rejected.

In [ ]:
def speculative_sampling_step(
    draft_probs: torch.Tensor,   # [K, vocab_size] — draft model's distributions for K proposed positions
    target_probs: torch.Tensor,  # [K, vocab_size] — target model's distributions at those positions
    draft_tokens: torch.Tensor,  # [K] — tokens the draft model proposed
) -> tuple[torch.Tensor, int]:
    """
    Core speculative sampling: accept/reject draft tokens and return accepted sequence.
    
    Returns:
        accepted_tokens: tokens to append to the sequence
        n_accepted: number of draft tokens actually accepted
    """
    K = draft_tokens.shape[0]
    accepted = []
    n_accepted = 0
    
    for i in range(K):
        token = draft_tokens[i].item()
        p_token = target_probs[i, token].item()  # target prob of this specific token
        q_token = draft_probs[i, token].item()   # draft prob of this specific token
        
        # Acceptance probability: min(1, p/q)
        acceptance_prob = min(1.0, p_token / (q_token + 1e-10))
        
        if random.random() < acceptance_prob:
            # Accept this token
            accepted.append(token)
            n_accepted += 1
        else:
            # Reject: sample from adjusted distribution (p - q)+
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0.0)
            norm = adjusted.sum()
            if norm > 0:
                adjusted = adjusted / norm
                correction = torch.multinomial(adjusted, num_samples=1).item()
            else:
                # Fallback: sample from target directly
                correction = torch.multinomial(target_probs[i], num_samples=1).item()
            accepted.append(correction)
            # Stop after first rejection
            return torch.tensor(accepted, dtype=torch.long), n_accepted
    
    # All K tokens accepted! Sample one bonus token from target at position K
    bonus = torch.multinomial(target_probs[-1], num_samples=1).item()  
    # Note: in practice we'd need a K+1-th target forward pass for the bonus;
    # here we reuse the last position's distribution as an approximation
    accepted.append(bonus)
    
    return torch.tensor(accepted, dtype=torch.long), n_accepted


# Quick sanity check: if draft == target, every token should be accepted
vocab_size = 100
K = 4
# Draft and target agree perfectly
identical_probs = F.softmax(torch.randn(K, vocab_size), dim=-1)
draft_tokens = torch.multinomial(identical_probs, num_samples=1).squeeze(-1)

accepted_tokens, n_acc = speculative_sampling_step(identical_probs, identical_probs, draft_tokens)
print(f"When draft == target: accepted {n_acc}/{K} draft tokens (+ bonus = {len(accepted_tokens)} total)")
print(f"Expected: all {K} accepted")

# Now test with very different distributions
target_probs_diff = F.softmax(torch.randn(K, vocab_size), dim=-1)
accepted_tokens2, n_acc2 = speculative_sampling_step(identical_probs, target_probs_diff, draft_tokens)
print(f"\nWhen draft ≠ target: accepted {n_acc2}/{K} draft tokens")

## Part 2: Building Toy Language Models

We need two models with different quality/speed profiles. For this demo:

- **Draft model ("small")**: A bigram model — fast, dumb, wrong a lot
- **Target model ("large")**: A trigram model — slower, smarter, more accurate

Both operate over a toy vocabulary with a known ground-truth distribution.
The key property we want: draft model is *correlated* with the target — it's often right,
just not always. This is what makes speculative decoding useful in practice.

In [ ]:
class BigramDraftModel:
    """
    A bigram language model: P(next | prev).
    Represents the 'small' draft model in speculative decoding.
    
    In practice this would be a smaller transformer (e.g., 7B drafting for 70B target).
    The key property: fast and often correct, but not always.
    """
    def __init__(self, vocab_size: int, smoothing: float = 0.1):
        self.vocab_size = vocab_size
        # Random bigram table — each row is P(next_token | prev_token)
        raw = torch.rand(vocab_size, vocab_size) + smoothing
        self.transition = raw / raw.sum(dim=-1, keepdim=True)
        self.forward_calls = 0
        self.simulated_latency_ms = 0.5  # fast
    
    def get_distribution(self, context: list[int]) -> torch.Tensor:
        """Returns P(next | context) as [vocab_size] tensor."""
        self.forward_calls += 1
        prev_token = context[-1] if context else 0
        return self.transition[prev_token].clone()
    
    def generate_draft(self, context: list[int], K: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Generate K draft tokens autoregressively.
        
        Returns:
            draft_tokens: [K] proposed tokens
            draft_probs: [K, vocab_size] distributions at each position
        """
        tokens = []
        probs = []
        current_context = list(context)
        
        for _ in range(K):
            dist = self.get_distribution(current_context)
            token = torch.multinomial(dist, num_samples=1).item()
            tokens.append(token)
            probs.append(dist)
            current_context.append(token)
        
        return torch.tensor(tokens), torch.stack(probs)
    
    def reset_stats(self):
        self.forward_calls = 0


class TrigramTargetModel:
    """
    A trigram language model: P(next | prev2, prev).
    Represents the 'large' target model in speculative decoding.
    
    We deliberately make this correlated with the draft model
    (same base distribution + noise) to simulate realistic draft-target alignment.
    """
    def __init__(self, draft_model: BigramDraftModel, noise_scale: float = 0.3):
        V = draft_model.vocab_size
        self.vocab_size = V
        self.draft_model = draft_model
        self.noise_scale = noise_scale
        # Per-bigram refinements
        self.refinements = torch.randn(V, V, V) * noise_scale
        self.forward_calls = 0
        self.simulated_latency_ms = 5.0  # 10x slower than draft
    
    def get_distribution(self, context: list[int]) -> torch.Tensor:
        """Returns P(next | context) as [vocab_size] tensor.
        
        Uses the draft model's bigram as a base, then adjusts based on trigram context.
        This creates a target that is *correlated* with (but not identical to) the draft.
        """
        self.forward_calls += 1
        # Start from draft model's distribution
        base = self.draft_model.transition[context[-1]].clone()
        
        # Add trigram refinement if we have enough context
        if len(context) >= 2:
            prev2 = context[-2]
            prev1 = context[-1]
            # Add a refinement that depends on the pair (prev2, prev1)
            logit_adjustment = self.refinements[prev2, prev1]
            # Apply refinement in log space
            log_base = torch.log(base + 1e-10) + logit_adjustment
            return F.softmax(log_base, dim=-1)
        
        return base
    
    def get_distributions_batch(self, context: list[int], draft_tokens: torch.Tensor) -> torch.Tensor:
        """
        Key efficiency gain: evaluate ALL K draft positions in one 'forward pass'.
        This is the parallel verification that makes speculative decoding fast.
        
        In a real transformer, this is a single forward pass with a K-length extension.
        The target model only needs to do ONE forward pass instead of K sequential ones.
        """
        self.forward_calls += 1  # Counts as ONE call, not K
        dists = []
        current_context = list(context)
        
        for token in draft_tokens:
            dist = self.get_distribution(current_context)
            dists.append(dist)
            current_context.append(token.item())
        
        return torch.stack(dists)
    
    def reset_stats(self):
        self.forward_calls = 0


# Create our models
VOCAB_SIZE = 50
draft = BigramDraftModel(VOCAB_SIZE)
target = TrigramTargetModel(draft, noise_scale=0.3)

# Sanity check
context = [5, 12, 7]
q_dist = draft.get_distribution(context)
p_dist = target.get_distribution(context)

# Measure correlation between draft and target
correlation = torch.dot(q_dist, p_dist) / (q_dist.norm() * p_dist.norm())
print(f"Draft-target distribution cosine similarity: {correlation:.3f}")
print(f"(1.0 = identical distributions, 0.0 = unrelated)")
print(f"Top-5 tokens (draft): {q_dist.topk(5).indices.tolist()}")
print(f"Top-5 tokens (target): {p_dist.topk(5).indices.tolist()}")

## Part 3: Standard Autoregressive Decoding (Baseline)

Before implementing speculative decoding, let's establish the baseline: pure target-model generation.
This is what we're trying to beat.

**Cost:** N tokens = N sequential target model forward passes.

In [ ]:
def autoregressive_generate(
    target_model: TrigramTargetModel,
    context: list[int],
    n_tokens: int,
    temperature: float = 1.0,
) -> tuple[list[int], int]:
    """
    Standard autoregressive generation: one forward pass per token.
    
    Returns:
        generated_tokens: the generated sequence
        n_forward_passes: number of target model calls
    """
    generated = []
    current_context = list(context)
    target_model.reset_stats()
    
    for _ in range(n_tokens):
        dist = target_model.get_distribution(current_context)
        if temperature != 1.0:
            dist = F.softmax(torch.log(dist + 1e-10) / temperature, dim=-1)
        token = torch.multinomial(dist, num_samples=1).item()
        generated.append(token)
        current_context.append(token)
    
    return generated, target_model.forward_calls


# Baseline
N_TOKENS = 50
context = [1, 2, 3]

tokens, n_calls = autoregressive_generate(target, context, N_TOKENS)
print(f"Standard autoregressive generation:")
print(f"  Tokens generated: {N_TOKENS}")
print(f"  Target model forward passes: {n_calls}")
print(f"  Forward passes per token: {n_calls / N_TOKENS:.1f}")
print(f"  Simulated latency: {n_calls * target.simulated_latency_ms:.0f}ms")

## Part 4: Speculative Decoding

Now the main event. The speculative loop:

```
While tokens_needed > 0:
  1. Draft model generates K tokens autoregressively (K cheap forward passes)
  2. Target model evaluates all K positions in ONE forward pass (parallel verification)
  3. Accept/reject draft tokens using the sampling algorithm
  4. Append accepted tokens (+ at least 1 correction token) to sequence
```

The speedup comes from step 2: one target model call replaces what would have been
up to K target model calls in standard decoding.

In [ ]:
@dataclass
class SpeculativeStats:
    """Tracking statistics for a speculative decoding run."""
    total_draft_calls: int = 0
    total_target_calls: int = 0
    total_tokens_generated: int = 0
    total_draft_tokens_accepted: int = 0
    total_draft_tokens_proposed: int = 0
    speculation_rounds: int = 0
    
    @property
    def acceptance_rate(self) -> float:
        if self.total_draft_tokens_proposed == 0:
            return 0.0
        return self.total_draft_tokens_accepted / self.total_draft_tokens_proposed
    
    @property 
    def tokens_per_target_call(self) -> float:
        if self.total_target_calls == 0:
            return 0.0
        return self.total_tokens_generated / self.total_target_calls
    
    @property
    def theoretical_speedup(self) -> float:
        """Target calls saved vs. autoregressive baseline."""
        baseline_calls = self.total_tokens_generated
        return baseline_calls / max(1, self.total_target_calls)


def speculative_generate(
    draft_model: BigramDraftModel,
    target_model: TrigramTargetModel,
    context: list[int],
    n_tokens: int,
    K: int = 4,  # speculation lookahead: draft K tokens per round
    temperature: float = 1.0,
) -> tuple[list[int], SpeculativeStats]:
    """
    Speculative decoding: draft K tokens, verify with target in one pass.
    
    Args:
        K: speculation length — how many tokens the draft model proposes each round.
           Larger K = more potential speedup but also more wasted draft work on rejection.
    
    Returns:
        generated_tokens: the generated sequence (from target distribution!)
        stats: profiling info
    """
    generated = []
    current_context = list(context)
    stats = SpeculativeStats()
    draft_model.reset_stats()
    target_model.reset_stats()
    
    while len(generated) < n_tokens:
        tokens_remaining = n_tokens - len(generated)
        k = min(K, tokens_remaining)
        
        # ── Step 1: Draft model generates K tokens (K cheap forward passes) ──
        draft_tokens, draft_probs = draft_model.generate_draft(current_context, k)
        stats.total_draft_calls += k
        stats.total_draft_tokens_proposed += k
        
        # ── Step 2: Target model verifies ALL K positions in ONE forward pass ──
        # This is the key parallelism! One call instead of K sequential calls.
        target_probs = target_model.get_distributions_batch(current_context, draft_tokens)
        stats.total_target_calls += 1
        stats.speculation_rounds += 1
        
        # Apply temperature if needed
        if temperature != 1.0:
            draft_probs = F.softmax(torch.log(draft_probs + 1e-10) / temperature, dim=-1)
            target_probs = F.softmax(torch.log(target_probs + 1e-10) / temperature, dim=-1)
        
        # ── Step 3: Accept/reject draft tokens ──
        accepted_tokens, n_accepted = speculative_sampling_step(
            draft_probs, target_probs, draft_tokens
        )
        stats.total_draft_tokens_accepted += n_accepted
        
        # ── Step 4: Append accepted tokens ──
        tokens_to_add = accepted_tokens[:tokens_remaining].tolist()
        generated.extend(tokens_to_add)
        current_context.extend(tokens_to_add)
        stats.total_tokens_generated += len(tokens_to_add)
    
    return generated, stats


# Run speculative decoding
generated_spec, stats = speculative_generate(draft, target, context, N_TOKENS, K=4)

print(f"Speculative decoding (K=4):")
print(f"  Tokens generated: {stats.total_tokens_generated}")
print(f"  Target model forward passes: {stats.total_target_calls}")
print(f"  Draft tokens proposed: {stats.total_draft_tokens_proposed}")
print(f"  Draft tokens accepted: {stats.total_draft_tokens_accepted}")
print(f"  Acceptance rate: {stats.acceptance_rate:.1%}")
print(f"  Tokens per target call: {stats.tokens_per_target_call:.2f}")
print(f"  Theoretical speedup: {stats.theoretical_speedup:.2f}x")
print(f"  Simulated latency: {stats.total_target_calls * target.simulated_latency_ms:.0f}ms")
print(f"  vs. baseline: {N_TOKENS * target.simulated_latency_ms:.0f}ms")

## Part 5: The Acceptance Rate Formula

How often will the draft model's tokens be accepted? It depends on how similar the two distributions are.

For a token $x$ sampled from the draft distribution $q$:

$$\mathbb{E}[\text{accepted}] = \sum_x q(x) \cdot \min\left(1, \frac{p(x)}{q(x)}\right) = \sum_x \min(q(x), p(x))$$

This is the **total variation overlap** between $p$ and $q$: $1 - \text{TV}(p, q)$.

So **acceptance rate = 1 - Total Variation Distance between draft and target distributions**.

When they're the same ($\text{TV} = 0$): 100% acceptance.  
When they're completely different ($\text{TV} = 1$): 0% acceptance.

Let's verify this empirically.

In [ ]:
def tv_distance(p: torch.Tensor, q: torch.Tensor) -> float:
    """Total variation distance: 0.5 * sum |p_i - q_i|"""
    return 0.5 * (p - q).abs().sum().item()


def expected_acceptance_rate(p: torch.Tensor, q: torch.Tensor) -> float:
    """Theoretical acceptance rate: sum_x min(p(x), q(x))"""
    return torch.minimum(p, q).sum().item()


def empirical_acceptance_rate(
    p: torch.Tensor, q: torch.Tensor, n_trials: int = 10000
) -> float:
    """Estimate acceptance rate by simulation."""
    tokens = torch.multinomial(q.unsqueeze(0).expand(n_trials, -1), num_samples=1).squeeze()
    p_vals = p[tokens]
    q_vals = q[tokens]
    acceptance_probs = torch.minimum(torch.ones_like(p_vals), p_vals / (q_vals + 1e-10))
    uniform = torch.rand(n_trials)
    return (uniform < acceptance_probs).float().mean().item()


# Sweep over different levels of draft-target alignment
print("Verifying acceptance rate = 1 - TV(p, q)")
print(f"{'Noise Scale':>12} {'TV Distance':>12} {'Expected Acc':>13} {'Empirical Acc':>14}")
print("-" * 55)

vocab_size = 50
base_dist = F.softmax(torch.randn(vocab_size) * 2, dim=-1)  # peaky base

for noise_scale in [0.0, 0.1, 0.3, 0.5, 0.8, 1.5, 3.0]:
    # Create target as base + noise
    noise = torch.randn(vocab_size) * noise_scale
    target_dist = F.softmax(torch.log(base_dist + 1e-10) + noise, dim=-1)
    
    tv = tv_distance(base_dist, target_dist)
    expected = expected_acceptance_rate(target_dist, base_dist)
    empirical = empirical_acceptance_rate(target_dist, base_dist)
    
    print(f"{noise_scale:>12.1f} {tv:>12.3f} {expected:>13.3f} {empirical:>14.3f}")

## Part 6: Speedup as a Function of Acceptance Rate

The expected number of tokens generated per speculation round (with draft length $K$) is:

$$\mathbb{E}[\text{tokens per round}] = \sum_{n=0}^{K} (n+1) \cdot \alpha^n \cdot (1 - \alpha) + (K+1) \cdot \alpha^K$$

where $\alpha$ is the per-step acceptance rate.

This simplifies to: $\dfrac{1 - \alpha^{K+1}}{1 - \alpha}$

And the theoretical wall-clock speedup (assuming target model cost $T$ dominates) is:

$$\text{Speedup} = \frac{\mathbb{E}[\text{tokens per round}]}{1} = \frac{1 - \alpha^{K+1}}{1 - \alpha}$$

(One target forward pass per round, so we get that many tokens for "free" relative to the baseline.)

In [ ]:
def expected_tokens_per_round(alpha: float, K: int) -> float:
    """Expected tokens generated per speculation round.
    
    Formula: (1 - alpha^(K+1)) / (1 - alpha)   for alpha < 1
    Formula: K + 1                               for alpha = 1 (all accepted + bonus)
    """
    if alpha >= 1.0 - 1e-8:
        return K + 1.0
    return (1 - alpha ** (K + 1)) / (1 - alpha)


# Plot speedup curves
alphas = np.linspace(0, 1, 200)
K_values = [1, 2, 4, 8, 16]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: tokens per round vs acceptance rate
ax = axes[0]
for K in K_values:
    speedups = [expected_tokens_per_round(a, K) for a in alphas]
    ax.plot(alphas, speedups, label=f"K={K}")

ax.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label="Baseline (no speculation)")
ax.set_xlabel("Acceptance Rate α", fontsize=12)
ax.set_ylabel("Expected Tokens per Target Call", fontsize=12)
ax.set_title("Speculative Decoding Throughput", fontsize=13)
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 12)
ax.grid(alpha=0.3)

# Right: heatmap of speedup over (alpha, K)
ax2 = axes[1]
K_range = np.arange(1, 17)
alpha_range = np.linspace(0.1, 0.99, 50)
speedup_grid = np.array([
    [expected_tokens_per_round(a, k) for a in alpha_range]
    for k in K_range
])

im = ax2.imshow(
    speedup_grid, aspect='auto', cmap='RdYlGn',
    extent=[alpha_range[0], alpha_range[-1], K_range[-1]+0.5, K_range[0]-0.5],
    vmin=1, vmax=8
)
plt.colorbar(im, ax=ax2, label="Expected tokens per target call")
ax2.set_xlabel("Acceptance Rate α", fontsize=12)
ax2.set_ylabel("Speculation Length K", fontsize=12)
ax2.set_title("Speedup Heatmap (K, α)", fontsize=13)

# Annotate a few key points
for a, k, label in [(0.8, 4, "typical\n(2-3x)"), (0.95, 8, "ideal\n(5-8x)"), (0.5, 4, "poor\n(~1.5x)")]:
    ax2.annotate(label, xy=(a, k), ha='center', va='center',
                fontsize=8, color='black',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('speculative_speedup.png', dpi=120, bbox_inches='tight')
plt.show()

# Key numbers
print("Expected tokens per target call at common working points:")
for alpha in [0.5, 0.7, 0.8, 0.9, 0.95]:
    for K in [4, 8]:
        e = expected_tokens_per_round(alpha, K)
        print(f"  α={alpha:.2f}, K={K}: {e:.2f}x")

## Part 7: Empirical Acceptance Rate Across Noise Levels

Let's run full speculative decoding experiments across a range of draft-target alignments
and measure the real acceptance rates and speedups. This tells us when speculative decoding is worth it.

In [ ]:
def run_spec_experiment(
    vocab_size: int = 50,
    noise_scale: float = 0.3,
    K: int = 4,
    n_tokens: int = 200,
    n_trials: int = 5,
) -> dict:
    """Run a speculative decoding experiment with given parameters."""
    acceptance_rates = []
    speedups = []
    
    for _ in range(n_trials):
        d = BigramDraftModel(vocab_size)
        t = TrigramTargetModel(d, noise_scale=noise_scale)
        context = list(torch.randint(0, vocab_size, (5,)).tolist())
        
        _, stats = speculative_generate(d, t, context, n_tokens, K=K)
        acceptance_rates.append(stats.acceptance_rate)
        speedups.append(stats.theoretical_speedup)
    
    return {
        'noise_scale': noise_scale,
        'K': K,
        'acceptance_rate': np.mean(acceptance_rates),
        'acceptance_rate_std': np.std(acceptance_rates),
        'speedup': np.mean(speedups),
        'speedup_std': np.std(speedups),
    }


print("Running experiments... (this may take a moment)")
results = []
for noise in [0.05, 0.1, 0.2, 0.4, 0.7, 1.0, 1.5, 2.0, 3.0]:
    for K in [2, 4, 8]:
        r = run_spec_experiment(noise_scale=noise, K=K, n_tokens=300, n_trials=3)
        results.append(r)

# Pivot into DataFrames for display
from collections import defaultdict
by_noise = defaultdict(dict)
for r in results:
    by_noise[r['noise_scale']][r['K']] = r

print(f"\n{'Noise':>7} | {'K=2 acc':>8} {'K=2 spd':>8} | {'K=4 acc':>8} {'K=4 spd':>8} | {'K=8 acc':>8} {'K=8 spd':>8}")
print("-" * 75)
for noise in sorted(by_noise.keys()):
    row = by_noise[noise]
    vals = []
    for K in [2, 4, 8]:
        if K in row:
            vals.append(f"{row[K]['acceptance_rate']:>8.1%}")
            vals.append(f"{row[K]['speedup']:>7.2f}x")
        else:
            vals.extend(["       -", "      -"])
    print(f"{noise:>7.2f} | {vals[0]} {vals[1]} | {vals[2]} {vals[3]} | {vals[4]} {vals[5]}")

## Part 8: Visualizing the Draft-Verify Loop

Let's visualize what happens during a single speculation round: which tokens
the draft proposes, and which ones survive the acceptance-rejection filter.

In [ ]:
def visualize_speculation_round(
    draft_model: BigramDraftModel,
    target_model: TrigramTargetModel,
    context: list[int],
    K: int = 6,
    n_rounds: int = 8,
):
    """Visualize which draft tokens get accepted/rejected across multiple rounds."""
    round_data = []  # list of (draft_tokens, accepted_mask, n_accepted)
    current_context = list(context)
    
    for _ in range(n_rounds):
        draft_tokens, draft_probs = draft_model.generate_draft(current_context, K)
        target_probs = target_model.get_distributions_batch(current_context, draft_tokens)
        accepted_tokens, n_accepted = speculative_sampling_step(draft_probs, target_probs, draft_tokens)
        
        # Build accept/reject mask
        accepted_mask = [True] * n_accepted + [False] * (K - n_accepted)
        round_data.append((draft_tokens.tolist(), accepted_mask, n_accepted))
        
        # Advance context with accepted tokens
        current_context.extend(accepted_tokens.tolist())
    
    # Plot
    fig, ax = plt.subplots(figsize=(13, 4))
    
    for round_idx, (tokens, mask, n_acc) in enumerate(round_data):
        for pos_idx, (token, accepted) in enumerate(zip(tokens, mask)):
            color = '#2ecc71' if accepted else '#e74c3c'  # green=accepted, red=rejected
            rect = mpatches.FancyBboxPatch(
                (round_idx * (K + 1) + pos_idx, 0.2), 0.85, 0.6,
                boxstyle="round,pad=0.05",
                facecolor=color, alpha=0.8, edgecolor='white', linewidth=1.5
            )
            ax.add_patch(rect)
            ax.text(
                round_idx * (K + 1) + pos_idx + 0.42, 0.5,
                str(token), ha='center', va='center', fontsize=9, color='white', fontweight='bold'
            )
        
        # Round label
        ax.text(
            round_idx * (K + 1) + K / 2 - 0.5, -0.1,
            f"Round {round_idx + 1}\n({n_acc}/{K} acc)",
            ha='center', va='top', fontsize=8, color='#555555'
        )
        
        # Separator
        if round_idx < n_rounds - 1:
            ax.axvline(x=round_idx * (K + 1) + K + 0.1, color='#cccccc', linewidth=1, linestyle='--')
    
    # Legend
    legend_handles = [
        mpatches.Patch(color='#2ecc71', label='Accepted (from target dist)'),
        mpatches.Patch(color='#e74c3c', label='Rejected (correction sampled)'),
    ]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=9)
    ax.set_xlim(-0.5, n_rounds * (K + 1) - 0.5)
    ax.set_ylim(-0.4, 1.1)
    ax.axis('off')
    ax.set_title(
        f"Speculation Rounds: {n_rounds} rounds × K={K} draft tokens  "
        f"(overall acceptance = {sum(r[2] for r in round_data) / (n_rounds*K):.0%})",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig('speculation_rounds.png', dpi=120, bbox_inches='tight')
    plt.show()


context = [5, 12, 7, 3]
visualize_speculation_round(draft, target, context, K=6, n_rounds=8)

## Part 9: Choosing K — The Diminishing Returns Trap

Larger K means more potential tokens per round, but there are hidden costs:

1. **Draft model compute**: K cheap calls, but not free
2. **Target model context length**: The verification pass processes K tokens — memory grows
3. **Diminishing returns**: With acceptance rate α, expected useful tokens ∝ `(1-α^(K+1))/(1-α)`

Let's find the optimal K as a function of α and the draft/target cost ratio.

In [ ]:
def wall_clock_speedup(
    alpha: float,
    K: int,
    draft_cost: float = 1.0,   # relative cost of one draft forward pass
    target_cost: float = 10.0, # relative cost of one target forward pass
) -> float:
    """
    Real wall-clock speedup accounting for draft model overhead.
    
    Per speculation round cost = K * draft_cost + target_cost
    Expected tokens per round = (1 - alpha^(K+1)) / (1 - alpha)
    
    Baseline cost = target_cost per token (pure autoregressive)
    """
    e_tokens = expected_tokens_per_round(alpha, K)
    round_cost = K * draft_cost + target_cost
    cost_per_token_spec = round_cost / e_tokens
    cost_per_token_base = target_cost  # baseline: one target call per token
    return cost_per_token_base / cost_per_token_spec


def optimal_K(
    alpha: float,
    draft_cost: float = 1.0,
    target_cost: float = 10.0,
    K_range: range = range(1, 32),
) -> tuple[int, float]:
    """Find K that maximizes wall-clock speedup."""
    best_K = 1
    best_speedup = 0
    for K in K_range:
        s = wall_clock_speedup(alpha, K, draft_cost, target_cost)
        if s > best_speedup:
            best_speedup = s
            best_K = K
    return best_K, best_speedup


# Plot optimal K vs acceptance rate for different cost ratios
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

alphas_plot = np.linspace(0.3, 0.98, 100)
cost_ratios = [5, 10, 20, 50]  # target_cost / draft_cost

# Left: optimal K
ax = axes[0]
for ratio in cost_ratios:
    optimal_Ks = [optimal_K(a, draft_cost=1.0, target_cost=ratio)[0] for a in alphas_plot]
    ax.plot(alphas_plot, optimal_Ks, label=f"Cost ratio {ratio}x")

ax.set_xlabel("Acceptance Rate α", fontsize=12)
ax.set_ylabel("Optimal K (speculation length)", fontsize=12)
ax.set_title("Optimal K vs Acceptance Rate", fontsize=13)
ax.legend()
ax.grid(alpha=0.3)

# Right: max speedup
ax2 = axes[1]
for ratio in cost_ratios:
    max_speedups = [optimal_K(a, draft_cost=1.0, target_cost=ratio)[1] for a in alphas_plot]
    ax2.plot(alphas_plot, max_speedups, label=f"Cost ratio {ratio}x")

ax2.axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label="Baseline")
ax2.set_xlabel("Acceptance Rate α", fontsize=12)
ax2.set_ylabel("Wall-clock Speedup", fontsize=12)
ax2.set_title("Best Achievable Speedup", fontsize=13)
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle("Optimal Speculation Length and Speedup", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('optimal_K.png', dpi=120, bbox_inches='tight')
plt.show()

# Print decision table
print("Optimal K for common scenarios (target is 10x more expensive than draft):")
print(f"{'Alpha':>7} {'Opt K':>7} {'Speedup':>9}")
print("-" * 27)
for alpha in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95]:
    k, spd = optimal_K(alpha, draft_cost=1, target_cost=10)
    print(f"{alpha:>7.2f} {k:>7} {spd:>8.2f}x")

## Part 10: When Does Speculative Decoding Win (and Lose)?

Speculative decoding is powerful, but not always the right tool. Let's characterize
the conditions where it excels and where it hurts.

In [ ]:
win_lose_table = [
    # (Scenario, Impact, Why)
    ("Coding tasks (syntax-heavy)",       "✅ Big Win",      "Draft model predicts keywords/punctuation at α>0.8"),
    ("Repetitive/formulaic output",        "✅ Big Win",      "High acceptance rate for predictable tokens"),
    ("Summarization (extractive)",         "✅ Win",          "Extracts phrases from input → draft matches well"),
    ("Classification / Yes-No answers",    "⚠️ Neutral",     "Output is short — speculation overhead not worth it"),
    ("Creative writing (high temperature)","⚠️ Reduced gain","High entropy outputs lower acceptance rate"),
    ("Math reasoning (chain-of-thought)",  "✅ Win",          "Structured reasoning patterns are predictable"),
    ("Multilingual / rare languages",      "❌ Loses",        "Draft model (usually English-dominant) diverges from target"),
    ("Very high temperature (T > 1.5)",    "❌ Loses",        "Distributions diverge — acceptance rate collapses"),
    ("Small target model (< 7B)",          "⚠️ Marginal",    "Target is cheap enough that draft overhead dominates"),
    ("Large batch decoding",               "❌ Not applicable","Batch processing is already compute-bound, not bandwidth-bound"),
    ("Streaming / interactive chat",       "✅ Big Win",      "TTFT and latency dominate — speculative helps both"),
]

print("When to use speculative decoding:")
print("=" * 85)
print(f"{'Scenario':<40} {'Impact':<18} {'Why'}")
print("-" * 85)
for scenario, impact, why in win_lose_table:
    print(f"{scenario:<40} {impact:<18} {why}")

print("\n" + "=" * 85)
print("Rule of thumb: speculative decoding wins when:")
print("  1. The large model is >> 7B (bandwidth-bound decoding)")
print("  2. A good draft model exists (same family, 7B vs 70B)")
print("  3. Temperature is low-moderate (< 1.0)")
print("  4. Outputs are predictable / structured")
print("  5. You're serving single-request or low-batch scenarios")

## Part 11: Quality Preservation — The Correctness Proof (Illustrated)

The most remarkable property of speculative decoding: **the output distribution is *exactly* the
target model's distribution**. Not approximately. Exactly.

This is non-obvious. Let's verify it empirically by comparing the token frequency distributions
from pure target-model generation vs. speculative generation over many samples.

In [ ]:
def measure_output_distribution(
    generate_fn,
    context: list[int],
    vocab_size: int,
    n_tokens: int = 5000,
) -> torch.Tensor:
    """Empirically measure the output token distribution by generating many tokens."""
    counts = torch.zeros(vocab_size)
    tokens = generate_fn(context, n_tokens)
    for t in tokens:
        counts[t] += 1
    return counts / counts.sum()


# Create fresh models for a clean test
torch.manual_seed(123)
V = 30  # small vocab for cleaner visualization
d_test = BigramDraftModel(V, smoothing=0.5)
t_test = TrigramTargetModel(d_test, noise_scale=0.5)
test_context = [1, 2, 3]

# Autoregressive baseline distribution
def ar_generate(ctx, n): return autoregressive_generate(t_test, ctx, n)[0]

# Speculative generation distribution
def spec_gen(ctx, n): return speculative_generate(d_test, t_test, ctx, n, K=4)[0]

print("Generating samples... (measuring output distributions)")
ar_dist = measure_output_distribution(ar_generate, test_context, V, n_tokens=3000)
spec_dist = measure_output_distribution(spec_gen, test_context, V, n_tokens=3000)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x = np.arange(V)
width = 0.4

ax = axes[0]
ax.bar(x - width/2, ar_dist.numpy(), width, label='Standard (target-only)', color='#3498db', alpha=0.8)
ax.bar(x + width/2, spec_dist.numpy(), width, label='Speculative', color='#e67e22', alpha=0.8)
ax.set_xlabel("Token ID")
ax.set_ylabel("Frequency")
ax.set_title("Output Distribution Comparison")
ax.legend()
ax.grid(alpha=0.3, axis='y')

# Difference plot
ax2 = axes[1]
diff = (spec_dist - ar_dist).numpy()
colors = ['#e74c3c' if d > 0 else '#2ecc71' for d in diff]
ax2.bar(x, diff, color=colors, alpha=0.8)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel("Token ID")
ax2.set_ylabel("Speculative - Standard")
ax2.set_title("Distribution Difference (should be near-zero)")
ax2.grid(alpha=0.3, axis='y')

tv = 0.5 * np.abs(diff).sum()
ax2.text(0.98, 0.95, f"TV Distance: {tv:.4f}\n(should be ≈ 0)",
         transform=ax2.transAxes, ha='right', va='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('distribution_verification.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"\nDistribution verification:")
print(f"  Total Variation Distance (spec vs baseline): {tv:.4f}")
print(f"  Max absolute difference: {np.abs(diff).max():.4f}")
print(f"  (Near-zero confirms speculative decoding preserves the target distribution)")

## Part 12: Variants and Extensions

The draft-verify loop has spawned a rich family of variants. Here's a map of the landscape.

In [ ]:
variants = [
    (
        "Classic Speculative Decoding",
        "Chen et al. / Leviathan et al. 2023",
        "Separate small draft model proposes tokens; large target verifies",
        "High quality; theoretically exact\nNeeds separate draft model deployment",
        "2–3x",
    ),
    (
        "Medusa",
        "Cai et al. 2024 (Georgia Tech)",
        "Add multiple speculative heads to the target model itself. Each head predicts\ntokens at future positions directly from current hidden states",
        "No separate model needed; compact\nSlightly lower acceptance rate vs. full draft model",
        "2–3x",
    ),
    (
        "EAGLE",
        "Li et al. 2024",
        "Draft model operates on feature (hidden state) level rather than token level.\nCheaper and more accurate than token-level draft",
        "Better acceptance rates than Medusa\nMore complex to implement",
        "3–4x",
    ),
    (
        "SpecInfer / Tree Speculation",
        "Miao et al. 2024",
        "Draft model builds a *tree* of possibilities. Target verifies the whole tree\nin one pass; select the best path",
        "Higher coverage per round\nMore complex verification (tree attention)",
        "2–4x",
    ),
    (
        "Lookahead Decoding",
        "Fu et al. 2024",
        "No draft model! Use Jacobi iteration on n-gram cache to propose tokens\nfrom the target model itself in parallel",
        "Zero extra model needed\nLower speedup than draft-model approaches",
        "1.5–2x",
    ),
    (
        "Self-Speculative / SkipDecode",
        "Zhang et al. 2024",
        "Early-exit from some layers to produce a cheap 'draft'; full forward pass verifies",
        "Single model, no memory overhead\nSmall speedup vs. full speculative",
        "1.3–2x",
    ),
]

print("Speculative Decoding Family Tree")
print("=" * 95)
for name, citation, mechanism, tradeoffs, speedup in variants:
    print(f"\n{name} ({citation})")
    print(f"  Mechanism: {mechanism}")
    print(f"  Tradeoffs: {tradeoffs}")
    print(f"  Typical speedup: {speedup}")
    print("-" * 95)

## Part 13: Putting It Together — Simulation with Realistic Numbers

Let's simulate a realistic production scenario: serving a 70B model with a 7B draft model.
We'll model the actual latency math from hardware specs.

In [ ]:
# Realistic hardware-informed numbers
# H100 SXM: 3.35 TB/s HBM bandwidth, 989 TFLOPS BF16

@dataclass
class ModelSpec:
    name: str
    params_b: float         # parameter count in billions
    bytes_per_param: float  # 2 for BF16, 1 for INT8, etc.
    
    @property
    def memory_gb(self) -> float:
        return self.params_b * 1e9 * self.bytes_per_param / 1e9
    
    def decode_latency_ms(self, bandwidth_tbs: float = 3.35, n_gpus: int = 1) -> float:
        """Time to load all weights for one decode step (memory-bandwidth bound)."""
        effective_bw = bandwidth_tbs * n_gpus * 1e12  # bytes/sec
        return (self.memory_gb * 1e9) / effective_bw * 1000  # ms


models = {
    'Llama-3.1-7B-BF16':  ModelSpec('7B BF16',  7,  2),
    'Llama-3.1-7B-INT8':  ModelSpec('7B INT8',  7,  1),
    'Llama-3.1-70B-BF16': ModelSpec('70B BF16', 70, 2),
    'Llama-3.1-70B-INT8': ModelSpec('70B INT8', 70, 1),
}

print("Memory-bandwidth bound decode latency per token (H100 SXM):")
print(f"{'Model':<25} {'Weight GB':>10} {'1 GPU (ms)':>12} {'4 GPU (ms)':>12} {'8 GPU (ms)':>12}")
print("-" * 75)
for name, spec in models.items():
    print(f"{name:<25} {spec.memory_gb:>10.0f} {spec.decode_latency_ms(n_gpus=1):>12.1f} "
          f"{spec.decode_latency_ms(n_gpus=4):>12.1f} {spec.decode_latency_ms(n_gpus=8):>12.1f}")

# Speculative decoding simulation
print("\n" + "=" * 75)
print("Speculative decoding: 7B draft → 70B target (BF16, 4 GPUs each)")

draft_spec = models['Llama-3.1-7B-BF16']
target_spec = models['Llama-3.1-70B-BF16']

draft_latency = draft_spec.decode_latency_ms(n_gpus=1)   # 7B on 1 GPU
target_latency = target_spec.decode_latency_ms(n_gpus=4)  # 70B on 4 GPUs

print(f"\nDraft model per-token latency: {draft_latency:.1f}ms")
print(f"Target model per-token latency: {target_latency:.1f}ms")
print(f"Cost ratio (target/draft): {target_latency / draft_latency:.1f}x")

print(f"\nProjected speedups by acceptance rate (K=4):")
print(f"{'Alpha':>8} {'Tokens/Round':>14} {'Latency/Token':>15} {'Speedup':>9}")
print("-" * 50)

for alpha in [0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    K = 4
    e_tokens = expected_tokens_per_round(alpha, K)
    round_cost_ms = K * draft_latency + target_latency
    latency_per_token = round_cost_ms / e_tokens
    speedup = target_latency / latency_per_token
    print(f"{alpha:>8.2f} {e_tokens:>14.2f} {latency_per_token:>14.1f}ms {speedup:>8.2f}x")

## Summary

Speculative decoding is one of the most elegant ideas in the LLM inference stack:
it's fast *and* lossless — a rare combination in engineering.

### Key Takeaways

**The Core Insight**  
LLM decode is memory-bandwidth bound, not compute bound. One target forward pass with K
tokens costs nearly the same wall-clock time as one forward pass with 1 token — so we can
verify K draft proposals for "free" relative to the baseline.

**The Algorithm**  
1. Draft model proposes K tokens autoregressively (K cheap calls)  
2. Target model verifies all K positions in one parallel forward pass  
3. Accept token $i$ with probability $\min(1, p_i / q_i)$  
4. On first rejection: sample correction token from $(p - q)^+$ distribution  
5. Output distribution = **exactly** the target model's distribution  

**The Math**  
- Acceptance rate per token = $\sum_x \min(p(x), q(x)) = 1 - \text{TV}(p, q)$  
- Expected tokens per round = $(1 - \alpha^{K+1}) / (1 - \alpha)$  
- Practical speedup: 2–3× for typical use cases, up to 4× for very aligned pairs  

**When It Wins**  
- Large target model (bandwidth-bound decoding, typically > 30B)  
- Available draft model from same family (7B drafting for 70B target)  
- Low-moderate temperature (structured output, coding, summarization)  
- Low-batch / interactive serving  

**When It Loses**  
- High temperature creative generation (distributions diverge → acceptance collapses)  
- Small target models (compute-bound, not bandwidth-bound)  
- Large batch serving (already compute-bound; better to just run more requests)  
- Multilingual tasks where the draft model wasn't trained on those languages  

### What's Next (Notebook 11: Continuous Batching)

Speculative decoding helped us generate more tokens per unit time. But what happens when
you have *thousands of requests* arriving at different times, each with different lengths?
Static batching wastes GPU cycles waiting for slow requests. **Continuous batching** (also
called "iteration-level scheduling") solves this — and it's how vLLM, TGI, and TensorRT-LLM
achieve the throughput numbers you see in production benchmarks.